In [18]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.integrate import dblquad
import matplotlib.pyplot as plt

# Constants
w_0 = 5
n_0 = 1.000277
z_w = 50
alpha = 0.002 # From compensatedLaser.pdf
dndT = -7.7*10**(-4) / 770 * 0.002
gamma = 0.5
n = n_0
xi = 1

# Initial conditions
C_0 = 100
W_x_0 = W_y_0 = (w_0 * np.sqrt(n_0 * alpha)) / (np.sqrt(w_0**4 + (0-z_w)**2))
T_x_0 = T_y_0 = 0
X_0 = Y_0 = 0
F_x_0 = F_y_0 = (-0.5 * n_0 * alpha * (0-z_w)) / (w_0**4 + (0-z_w)**2)
P_0 = np.arctan(z_w / w_0**2)

# Storing initial conditions
y = C_0, W_x_0, W_y_0, T_x_0, T_y_0, X_0, Y_0, F_x_0, F_y_0, P_0

def heat_profile(y):  
    C_0, W_x_0, W_y_0, T_x_0, T_y_0, X_0, Y_0, F_x_0, F_y_0, P_0 = y
    # Define the (x, y) grid
    x_grid, y_grid = np.meshgrid(np.linspace(-100, 100, 100), np.linspace(-100, 100, 100))
    
    # Initialize temperature distribution
    T = np.zeros_like(x_grid)  
        
    # Solve the heat equation for each (x, y) in the grid
    for i in range(100):
        for j in range(100):
            x = x_grid[i, j]
            y = y_grid[i, j]
            # Solve the heat equation at (x, y) for the current z-step
            T[i, j], _ = solve_heat_equation(x, y, X_0, Y_0, W_x_0, W_y_0)
    return T

def solve_heat_equation(x, y, X, Y, W_x, W_y):  
    f = lambda x_prime, y_prime: (1/2 * np.log((x - x_prime)**2 + (y - y_prime)**2))
    g = lambda x_prime, y_prime: (np.exp(-W_x**2 * (x_prime - X)**2) * np.exp(-W_y**2 * (y_prime - Y)**2))
    integrand = lambda x_prime, y_prime: f(x_prime,y_prime) * g(x_prime,y_prime)
    
    result=  dblquad(integrand, X-1, X+1, Y-1, Y+1)
    return result

results = -dndT*heat_profile(y)

plt.imshow(results, cmap='gist_heat', interpolation='gaussian', origin="lower")
plt.colorbar(label='Value')
plt.title('Heat Profile')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()
results